# PolicyGuard-4B: 100 Test Cases Inference Pipeline

This notebook executes the minimal **PolicyGuard-4B** inference pipeline on the bundled 100 test cases located in `policyguard_prompts_100/policyguard_inputs`.

### Pipeline Highlights
- **Bundled Inputs:** Uses the local repository inputs in `policyguard_prompts_100/policyguard_inputs` (no Google Drive mounting needed).
- **Pretrained Model:** Loads `Rakancorle1/PolicyGuard-4B` with FP16 on GPU.
- **Greedy Decoding:** Deterministic token generation (`do_sample=False`, `max_new_tokens=4`).
- **Decision Extraction:** Extracts decisions directly into `violation`, `no_violation`, or `UNKNOWN`.
- **Streaming Resilience:** Writes and flushes each prediction to `predictions.jsonl`, `predictions.csv`, and `summary.json`.

## Cell 1: Environment Setup

> **GPU Check:** Ensure you have selected a GPU runtime (**Runtime -> Change runtime type -> T4 GPU or higher**).

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA is available! GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
else:
    print("WARNING: CUDA is NOT available. PolicyGuard inference requires a GPU runtime.")
    print("Please go to: Runtime -> Change runtime type -> Select T4 GPU or higher.")

# Install minimal required packages
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 safetensors>=0.4.0


## Cell 2: Repository Setup

Clones or navigates to the repository in the local environment and imports the pipeline module.

In [ ]:
import os
import sys
import importlib

# If running in a fresh Colab session, clone the repository
if not os.path.exists("policyguard_pipeline.py"):
    if not os.path.exists("GuardSLM"):
        os.system("git clone https://github.com/AkarshiAaryan/GuardSLM.git")
    if os.path.exists("GuardSLM"):
        os.chdir("GuardSLM")

# Ensure repository directory is in sys.path
if "." not in sys.path:
    sys.path.insert(0, ".")

# Automatically pull latest updates
if os.path.exists(".git"):
    os.system("git pull origin main")

# Force reload module in active Jupyter kernel to apply any code changes
import policyguard_pipeline
importlib.reload(policyguard_pipeline)

from policyguard_pipeline import (
    load_model,
    generate_response,
    parse_output,
    run_batch,
    evaluate_predictions,
    load_ground_truth
)
print("Successfully imported and reloaded latest policyguard_pipeline.")


## Cell 3: User Configuration

Sets directory paths and generation parameters using the bundled test cases.

In [ ]:
MODEL_ID = "Rakancorle1/PolicyGuard-4B"
INPUT_DIR = "policyguard_prompts_100/policyguard_inputs"
OUTPUT_DIR = "results/policyguard_predictions"
GROUND_TRUTH_PATH = "policyguard_evaluation_ground_truth.csv"
EXPECTED_CASES = 100
MAX_NEW_TOKENS = 4

print("Configuration Parameters:")
print(f"  Model ID:          {MODEL_ID}")
print(f"  Input Directory:   {INPUT_DIR}")
print(f"  Output Directory:  {OUTPUT_DIR}")
print(f"  Ground Truth File: {GROUND_TRUTH_PATH}")
print(f"  Expected Cases:    {EXPECTED_CASES}")
print(f"  Max New Tokens:    {MAX_NEW_TOKENS}")


## Cell 4: Input Validation

Verifies that `INPUT_DIR` exists and contains all 100 `.txt` prompt files.

In [ ]:
import re
import pandas as pd

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r"(\d+)", s)]

if not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(f"Input directory '{INPUT_DIR}' not found.")

txt_files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith(".txt")], key=natural_sort_key)
print(f"Found {len(txt_files)} .txt files in '{INPUT_DIR}'.")

assert len(txt_files) == EXPECTED_CASES, f"Expected {EXPECTED_CASES} files, but found {len(txt_files)}."
print(f"[VALIDATION PASSED] Exactly {EXPECTED_CASES} test case files detected.")

if os.path.exists(GROUND_TRUTH_PATH):
    gt_df = pd.read_csv(GROUND_TRUTH_PATH)
    print(f"[VALIDATION PASSED] Ground truth file '{GROUND_TRUTH_PATH}' loaded ({len(gt_df)} cases).")
    print(f"  - Actual Violations (BLOCK):    {(gt_df['expected_label'] == 'violation').sum()}")
    print(f"  - Actual No-Violations (ALLOW): {(gt_df['expected_label'] == 'no_violation').sum()}")
else:
    print(f"WARNING: Ground truth file '{GROUND_TRUTH_PATH}' not found.")

print("\nFirst 5 test cases:")
for f in txt_files[:5]:
    print(f"  - {f}")


## Cell 5: Load PolicyGuard-4B Model

Loads the pretrained tokenizer and model once into GPU memory.

In [ ]:
tokenizer, model = load_model(model_id=MODEL_ID)
print("PolicyGuard-4B model and tokenizer loaded successfully.")


## Cell 6: Smoke Test (3 Cases)

Executes 3 test cases into a separate test folder to verify the pipeline before the full run.

In [ ]:
smoke_test_dir = "results/smoke_test"

print("Running 3-case smoke test on bundled inputs...")
smoke_summary = run_batch(
    input_dir=INPUT_DIR,
    output_dir=smoke_test_dir,
    model=model,
    tokenizer=tokenizer,
    limit=3,
    resume=False,
    model_id=MODEL_ID,
    max_new_tokens=MAX_NEW_TOKENS
)
print("Smoke test completed successfully!")


## Cell 7: Full Experiment (All 100 Cases)

Executes all 100 test cases sequentially with streaming progress logging.

In [ ]:
print(f"Executing Full 100-Case Experiment on '{INPUT_DIR}'...")

# Note: With resume=True, any cached outputs are dynamically re-parsed with the latest parsing logic.
# To force fresh inference from scratch without using cached outputs, set resume=False.
full_summary = run_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    model=model,
    tokenizer=tokenizer,
    limit=None,
    resume=True,
    model_id=MODEL_ID,
    max_new_tokens=MAX_NEW_TOKENS,
    ground_truth_path=GROUND_TRUTH_PATH
)


## Cell 8: Output Collection & Inspection

Displays run operational statistics and previews the predictions CSV.

In [ ]:
import pandas as pd

print("=" * 60)
print("EXPERIMENT RUN SUMMARY")
print("=" * 60)
print(f"Total Inputs Evaluated:    {full_summary['number_of_inputs']}")
print(f"Successfully Generated:    {full_summary['successfully_generated_count']}")
print(f"Violations:                {full_summary['count_violation']}")
print(f"No Violations:             {full_summary['count_no_violation']}")
print(f"UNKNOWN Responses:         {full_summary['count_UNKNOWN']}")
print(f"Average Latency:           {full_summary['average_latency_ms']} ms")
print(f"Median Latency:            {full_summary['median_latency_ms']} ms")
print(f"Total Execution Time:      {full_summary['total_runtime_seconds']} s")
print("\nSaved Output Files:")
print(f"  JSONL:    {full_summary['predictions_jsonl_path']}")
print(f"  CSV:      {full_summary['predictions_csv_path']}")
print(f"  Summary:  {full_summary['summary_json_path']}")

csv_path = full_summary['predictions_csv_path']
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\nPredictions CSV Preview (First 10 rows):")
    display(df.head(10))


## Cell 9: Model Accuracy & Ground-Truth Evaluation

Evaluates the generated predictions against the expected labels in `policyguard_evaluation_ground_truth.csv`.
Computes overall accuracy, correct vs incorrect counts, confusion matrix, and lists any mismatched test cases.

In [ ]:
import json
import pandas as pd

# Load or run evaluation
eval_summary = full_summary.get("evaluation")
if not eval_summary:
    eval_summary = evaluate_predictions(
        predictions=full_summary["predictions_csv_path"],
        ground_truth_path=GROUND_TRUTH_PATH,
        output_dir=OUTPUT_DIR
    )

cm = eval_summary["confusion_matrix"]
cb = eval_summary["class_breakdown"]
total = eval_summary["total_test_cases"]
correct = eval_summary["correct_predictions"]
incorrect = eval_summary["incorrect_predictions"]
acc = eval_summary["accuracy_pct"]

print("=" * 65)
print("POLICYGUARD-4B BENCHMARK EVALUATION RESULTS")
print("=" * 65)
print(f"  Total Test Cases:       {total}")
print(f"  Correct Decisions:      {correct} / {total} ({acc}%)")
print(f"  Incorrect Decisions:    {incorrect} / {total} ({round(100.0 - acc, 2)}%)")
print("-" * 65)
print(f"  Violation Recall:       {cb['violation_recall_sensitivity']:.2%}  (Sensitivity - Caught Violations)")
print(f"  No-Violation Specific:  {cb['no_violation_specificity']:.2%}  (Specificity - Allowed Benign)")
print(f"  Precision:              {cb['violation_precision']:.2%}")
print(f"  F1-Score:               {cb['f1_score']:.4f}")
print("=" * 65)

print("\n--- Confusion Matrix ---")
cm_df = pd.DataFrame(
    [
        [cm["true_violations_tp"], cm["false_no_violations_fn"], cm["unknown_on_violation"], cb["actual_violations_count"]],
        [cm["false_violations_fp"], cm["true_no_violations_tn"], cm["unknown_on_no_violation"], cb["actual_no_violations_count"]]
    ],
    index=["Actual Violation (BLOCK)", "Actual No-Violation (ALLOW)"],
    columns=["Predicted Violation", "Predicted No-Violation", "Predicted UNKNOWN", "Total"]
)
display(cm_df)

# Preview Evaluation Results Table
eval_csv_path = eval_summary.get("evaluation_csv_path") or os.path.join(OUTPUT_DIR, "evaluation_results.csv")
if os.path.exists(eval_csv_path):
    eval_df = pd.read_csv(eval_csv_path)
    print("\n--- Evaluation Results Table (First 10 Cases) ---")
    display(eval_df.head(10))

# Display Mismatched Cases Table if any
mismatches = eval_summary.get("mismatched_cases", [])
if mismatches:
    mismatch_df = pd.DataFrame(mismatches)[
        ["case_id", "input_filename", "expected_label", "expected_decision", "predicted_label", "error_type", "raw_model_output"]
    ]
    print(f"\n--- Mismatched / Incorrect Test Cases ({len(mismatches)} Cases) ---")
    display(mismatch_df)
else:
    print("\nALL 100 TEST CASES PASSED PERFECTLY (0 MISMATCHES)!")

print("\nEvaluation Reports Saved:")
print(f"  - Report (Markdown): {eval_summary.get('evaluation_report_md_path')}")
print(f"  - Metrics (JSON):    {eval_summary.get('evaluation_json_path')}")
print(f"  - Results (CSV):     {eval_csv_path}")


## Cell 10: Cleanup

Safely releases model memory and empties the GPU cache.

In [ ]:
import gc

print("Releasing model and clearing GPU memory...")
del model
del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared successfully.")
print("Cleanup complete.")
